# **Notebook 02:** Limpeza e Tratamento de Dados
---

**Projeto:** Bellabeat Data Analysis (Kaggle FitBit Dataset)  
**Autor:** Alessandro Moreira Cecilio  
**Data:** Outubro / 2026  

### Objetivo
O objetivo deste notebook é realizar a limpeza e o tratamento do conjunto de dados consolidado da FitBit, preparando os dados para as etapas de análise exploratória, consultas SQL e visualização no Power BI.

Nesta etapa serão investigados possíveis problemas de qualidade e consistência dos dados, garantindo que as informações utilizadas nas análises sejam adequadas e confiáveis.

### Etapas

Neste notebook serão realizadas as seguintes etapas:

**1. Importação das bibliotecas necessárias;**<br>
**2. Carregamento do dataset consolidado;**<br>
**3. Inspeção Inicial**<br>
**4. Verificação dos tipos de dados;**<br>
**5. Conversão e padronização de variáveis;**<br>
**6. Identificação e tratamento de valores ausentes;**<br>
**7. Identificação e tratamento de registros duplicados;**<br>
**8. Verificação de inconsistências nos dados e valores inválidos;**<br>
**9. Criação de novas variáveis para análise;**<br>
**10. Identificação e tratamento de outliers;**<br>
**11. Validação do dataset após o tratamento;**<br>
**12. Salvamento do dataset processado.**

---

---

## 1. Importação das Bibliotecas

In [ ]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# Adiciona a raiz do projeto ao sys.path para conseguir importar a pasta src
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_table

PROCESSED_DATA_PATH = "../data/processed/bronze"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## 2. Carregamento do dataset

In [71]:
# Tabelas que serão carregadas
tabelas= [
    "dailyActivity_merged_concat",
    "sleepDay_merged_concat",
    "weightLogInfo_merged_concat",
    "hourlySteps_merged_concat"
]

In [72]:
dfs = {}

for tabela in tabelas:
    try:
        df = load_table(PROCESSED_DATA_PATH, tabela)

        # Remove o sufixo '_merged_concat' para usar como chave do dicionário
        nome_curto = tabela.replace("_merged_concat", "")

        # Adiciona no dicionário com o nome curto
        dfs[nome_curto] = df
        
        print(f"Tabela '{nome_curto}' carregada com sucesso!")
    except FileNotFoundError as e:
        print(e)

Tabela 'dailyActivity' carregada com sucesso!
Tabela 'sleepDay' carregada com sucesso!
Tabela 'weightLogInfo' carregada com sucesso!
Tabela 'hourlySteps' carregada com sucesso!


---

## 3. Inspeção Inicial

In [73]:
# Lista de dataframes carregados
dataframes = [
    "dailyActivity",
    "sleepDay",
    "weightLogInfo",
    "hourlySteps"
]

In [74]:
print(" ====== Estrutura dos DataFrames: =======\n")
for df_nome in dataframes:
    if df_nome in dfs:
        print(f"DataFrame '{df_nome}':")
        print(dfs[df_nome].shape[0], "linhas x", dfs[df_nome].shape[1], "colunas")
        print(dfs[df_nome].info())
        print("\n" + "=" * 100 + "\n")
    else:
        print(f"DataFrame '{df_nome}' não foi carregado.")

 ====== Estrutura dos DataFrames: =======

DataFrame 'dailyActivity':
1397 linhas x 16 colunas
<class 'pandas.DataFrame'>
RangeIndex: 1397 entries, 0 to 1396
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Id                        1397 non-null   int64  
 1   ActivityDate              1397 non-null   str    
 2   TotalSteps                1397 non-null   int64  
 3   TotalDistance             1397 non-null   float64
 4   TrackerDistance           1397 non-null   float64
 5   LoggedActivitiesDistance  1397 non-null   float64
 6   VeryActiveDistance        1397 non-null   float64
 7   ModeratelyActiveDistance  1397 non-null   float64
 8   LightActiveDistance       1397 non-null   float64
 9   SedentaryActiveDistance   1397 non-null   float64
 10  VeryActiveMinutes         1397 non-null   int64  
 11  FairlyActiveMinutes       1397 non-null   int64  
 12  LightlyActiveMinutes      1397 non

In [75]:
print(" ====== Amostras dos DataFrames: ====== \n")
for df_nome in dataframes:
    if df_nome in dfs:
        print(f"DataFrame '{df_nome}':")
        display(dfs[df_nome].head())
        print("\n" + "=" * 100 + "\n")
    else:
        print(f"DataFrame '{df_nome}' não foi carregado.")

 ====== Amostras dos DataFrames: ====== 

DataFrame 'dailyActivity':


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,source_folder
0,1503960366,4/12/2016,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,25,13,328,728,1985,Fitabase Data 4.12.16-5.12.16
1,1503960366,4/13/2016,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,21,19,217,776,1797,Fitabase Data 4.12.16-5.12.16
2,1503960366,4/14/2016,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,30,11,181,1218,1776,Fitabase Data 4.12.16-5.12.16
3,1503960366,4/15/2016,9762,6.28,6.28,0.0,2.14,1.26,2.83,0.0,29,34,209,726,1745,Fitabase Data 4.12.16-5.12.16
4,1503960366,4/16/2016,12669,8.16,8.16,0.0,2.71,0.41,5.04,0.0,36,10,221,773,1863,Fitabase Data 4.12.16-5.12.16




DataFrame 'sleepDay':


,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed,source_folder
0,1503960366,4/12/2016 12:00:00 AM,1,327,346,Fitabase Data 4.12.16-5.12.16
1,1503960366,4/13/2016 12:00:00 AM,2,384,407,Fitabase Data 4.12.16-5.12.16
2,1503960366,4/15/2016 12:00:00 AM,1,412,442,Fitabase Data 4.12.16-5.12.16
3,1503960366,4/16/2016 12:00:00 AM,2,340,367,Fitabase Data 4.12.16-5.12.16
4,1503960366,4/17/2016 12:00:00 AM,1,700,712,Fitabase Data 4.12.16-5.12.16




DataFrame 'weightLogInfo':


,Id,Date,WeightKg,WeightPounds,Fat,BMI,IsManualReport,LogId,source_folder
0,1503960366,5/2/2016 11:59:59 PM,52.599998,115.963147,22.0,22.650000,True,1462233599000,Fitabase Data 4.12.16-5.12.16
1,1503960366,5/3/2016 11:59:59 PM,52.599998,115.963147,NaN,22.650000,True,1462319999000,Fitabase Data 4.12.16-5.12.16
2,1927972279,4/13/2016 1:08:52 AM,133.500000,294.317120,NaN,47.540001,False,1460509732000,Fitabase Data 4.12.16-5.12.16
3,2873212765,4/21/2016 11:59:59 PM,56.700001,125.002104,NaN,21.450001,True,1461283199000,Fitabase Data 4.12.16-5.12.16
4,2873212765,5/12/2016 11:59:59 PM,57.299999,126.324875,NaN,21.690001,True,1463097599000,Fitabase Data 4.12.16-5.12.16




DataFrame 'hourlySteps':


,Id,ActivityHour,StepTotal,source_folder
0,1503960366,4/12/2016 12:00:00 AM,373,Fitabase Data 4.12.16-5.12.16
1,1503960366,4/12/2016 1:00:00 AM,160,Fitabase Data 4.12.16-5.12.16
2,1503960366,4/12/2016 2:00:00 AM,151,Fitabase Data 4.12.16-5.12.16
3,1503960366,4/12/2016 3:00:00 AM,0,Fitabase Data 4.12.16-5.12.16
4,1503960366,4/12/2016 4:00:00 AM,0,Fitabase Data 4.12.16-5.12.16


In [76]:
print(" ====== Descrição Estatística dos DataFrames: ====== \n")
for df_nome in dataframes:
    if df_nome in dfs:
        print(f"DataFrame '{df_nome}':")
        print(dfs[df_nome].describe())
        print("\n" + "=" * 100 + "\n")
    else:
        print(f"DataFrame '{df_nome}' não foi carregado.")

 ====== Descrição Estatística dos DataFrames: ====== 

DataFrame 'dailyActivity':
                 Id    TotalSteps  TotalDistance  TrackerDistance  \
count  1.397000e+03   1397.000000    1397.000000      1397.000000   
mean   4.781210e+09   7280.898354       5.219434         5.192219   
std    2.384293e+09   5214.336113       3.994206         3.980077   
min    1.503960e+09      0.000000       0.000000         0.000000   
25%    2.320127e+09   3146.000000       2.170000         2.160000   
50%    4.445115e+09   6999.000000       4.950000         4.950000   
75%    6.962181e+09  10544.000000       7.500000         7.480000   
max    8.877689e+09  36019.000000      28.030001        28.030001   

       LoggedActivitiesDistance  VeryActiveDistance  ModeratelyActiveDistance  \
count               1397.000000         1397.000000               1397.000000   
mean                   0.131481            1.397416                  0.538461   
std                    0.703683            2.607480  

---

## 4. Verificação dos tipos de dados

In [77]:
print("====== Tipos de Dados dos DataFrames ======\n")

for df_nome, df in dfs.items():
    if df_nome in dfs:
        print(f"DataFrame: '{df_nome}'")
        print(df.dtypes)
        print("\n" + "-" * 50 + "\n")
    else:
        print(f"DataFrame '{df_nome}' não foi carregado.")

====== Tipos de Dados dos DataFrames ======

DataFrame: 'dailyActivity'
Id                            int64
ActivityDate                    str
TotalSteps                    int64
TotalDistance               float64
TrackerDistance             float64
LoggedActivitiesDistance    float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
source_folder                   str
dtype: object

--------------------------------------------------

DataFrame: 'sleepDay'
Id                    int64
SleepDay                str
TotalSleepRecords     int64
TotalMinutesAsleep    int64
TotalTimeInBed        int64
source_folder           str
dtype: object

--------------------------------------------------

DataFrame: 'weightLogInfo'
I

### Diagnóstico de Tipos e Plano de Padronização

A verificação dos tipos de dados (`dtypes`) e da estrutura do dicionário revelou as seguintes necessidades de ajuste para a consolidação da **Camada Silver**:

1. **Padronização dos Nomes das Tabelas (`CamelCase` $\rightarrow$ `snake_case`):**
   * As chaves do dicionário `dfs` serão remapeadas para `snake_case` (`daily_activity`, `sleep_day`, `hourly_steps`, `weight_log`).
   * Isso garante consistência no código e previne problemas de sintaxe no **Oracle SQL**, evitando a necessidade de aspas duplas em identificadores no banco de dados.

2. **Conversão de Datas/Horas (`object` $\rightarrow$ `datetime64`):**
   * As variáveis temporais em todas as tabelas (`ActivityDate`, `SleepDay`, `ActivityHour` e `Date`) foram identificadas como texto (`object`). Serão convertidas para o tipo `datetime64` para viabilizar análises temporais e agregações no Python e Oracle SQL.
   * A coluna `ActivityHour` da tabela `hourlySteps` será mantida com sua granularidade de data e hora (`activity_hour`), pois servirá para análises de picos de comportamento ao longo do dia.

3. **Padronização de Nomes de Colunas (`CamelCase` $\rightarrow$ `snake_case`):**
   * Todas as colunas serão renomeadas para o padrão `snake_case` (letras minúsculas separadas por `_`), atendendo às boas práticas do PEP 8 e garantindo compatibilidade direta com o Oracle SQL.
   * Evita-se o uso do nome reservado `DATE` nas colunas de data pura (substituído por `activity_date`, `sleep_date`, `weight_date`).

4. **Demais Variáveis:**
   * Os tipos numéricos inteiros (`int64`) e decimais (`float64`), além do booleano (`bool`), já se encontram nos formatos adequados e preservam a integridade dos dados originais.

---

## 5. Conversão e padronização de variáveis

In [78]:
# Dicionário de mapeamento 
nome_map = {
    'dailyActivity': 'daily_activity',
    'sleepDay': 'sleep_day',
    'hourlySteps': 'hourly_steps',
    'weightLogInfo': 'weight_log'
}

# Re-criando o dicionário com as novas chaves em snake_case
dfs = {nome_map.get(k, k): v for k, v in dfs.items()}

In [79]:
#  Renomeando colunas da dailyActivity

dfs['daily_activity'].rename(columns={
    'Id': 'user_id',
    'ActivityDate': 'activity_date',
    'TotalSteps': 'total_steps',
    'TotalDistance': 'total_distance',
    'TrackerDistance': 'tracker_distance',
    'LoggedActivitiesDistance': 'logged_activities_distance',
    'VeryActiveDistance': 'very_active_distance',
    'ModeratelyActiveDistance': 'moderately_active_distance',
    'LightActiveDistance': 'light_active_distance',
    'SedentaryActiveDistance': 'sedentary_active_distance',
    'VeryActiveMinutes': 'very_active_minutes',
    'FairlyActiveMinutes': 'fairly_active_minutes',
    'LightlyActiveMinutes': 'lightly_active_minutes',
    'SedentaryMinutes': 'sedentary_minutes',
    'Calories': 'calories'
}, inplace=True)

In [80]:
# Renomeando colunas da sleepDay
dfs['sleep_day'].rename(columns={
    'Id': 'user_id',
    'SleepDay': 'sleep_date',
    'TotalSleepRecords': 'total_sleep_records',
    'TotalMinutesAsleep': 'total_minutes_asleep',
    'TotalTimeInBed': 'total_time_in_bed'
}, inplace=True)

In [81]:
# Renomeando colunas da hourlySteps
dfs['hourly_steps'].rename(columns={
    'Id': 'user_id',
    'ActivityHour': 'activity_hour',
    'StepTotal': 'step_total'
}, inplace=True)

In [82]:
# Renomeando colunas da weightLogInfo
dfs['weight_log'].rename(columns={
    'Id': 'user_id',
    'Date': 'weight_date',
    'WeightKg': 'weight_kg',
    'WeightPounds': 'weight_pounds',
    'Fat': 'fat_percentage',
    'BMI': 'bmi',
    'IsManualReport': 'is_manual_report',
    'LogId': 'log_id'
}, inplace=True)

#### Checando nome das colunas

In [88]:
novos_dataframes = list(dfs.keys())

In [89]:
print(" ====== Colunas dos DataFrames: =======\n")
for df_nome in novos_dataframes:
        print(f"DataFrame '{df_nome}':")
        print(dfs[df_nome].columns)
        print("\n" + "=" * 100 + "\n")

 ====== Colunas dos DataFrames: =======

DataFrame 'daily_activity':
Index(['user_id', 'activity_date', 'total_steps', 'total_distance',
       'tracker_distance', 'logged_activities_distance',
       'very_active_distance', 'moderately_active_distance',
       'light_active_distance', 'sedentary_active_distance',
       'very_active_minutes', 'fairly_active_minutes',
       'lightly_active_minutes', 'sedentary_minutes', 'calories',
       'source_folder'],
      dtype='str')


DataFrame 'sleep_day':
Index(['user_id', 'sleep_date', 'total_sleep_records', 'total_minutes_asleep',
       'total_time_in_bed', 'source_folder'],
      dtype='str')


DataFrame 'weight_log':
Index(['user_id', 'weight_date', 'weight_kg', 'weight_pounds',
       'fat_percentage', 'bmi', 'is_manual_report', 'log_id', 'source_folder'],
      dtype='str')


DataFrame 'hourly_steps':
Index(['user_id', 'activity_hour', 'step_total', 'source_folder'], dtype='str')




#### Convertendo colunas de datas

In [83]:
# Lista de colunas que representam datas em cada DataFrame
colunas_data = [
    'activity_date',
    'sleep_date',
    'activity_hour',
    'weight_date'
]

In [ ]:
# Convertendo colunas de data para o tipo datetime de forma segura
for df_nome, df in dfs.items():
    for coluna in colunas_data:
        if coluna in df.columns:
            df[coluna] = pd.to_datetime(df[coluna], format='mixed')
            print(f"[{df_nome}] Coluna '{coluna}' convertida para datetime com sucesso.")

[daily_activity] Coluna 'activity_date' convertida para datetime com sucesso.
[sleep_day] Coluna 'sleep_date' convertida para datetime com sucesso.
[weight_log] Coluna 'weight_date' convertida para datetime com sucesso.
[hourly_steps] Coluna 'activity_hour' convertida para datetime com sucesso.


#### Checando conversão

In [ ]:
# Relatório de tipos de dados após a conversão
for df_nome, df in dfs.items():
    for coluna in colunas_data:
        if coluna in df.columns:
            print(f"[{df_nome}] '{coluna}': {df[coluna].dtype}")

[daily_activity] 'activity_date': datetime64[us]
[sleep_day] 'sleep_date': datetime64[us]
[weight_log] 'weight_date': datetime64[us]
[hourly_steps] 'activity_hour': datetime64[us]


## 6. Identificação e tratamento de valores ausentes